# Exploratory Data Analysis (EDA) - Segmented by Review Origin

This notebook explores the preprocessed restaurant reviews dataset. It segments and analyzes reviews by their **Origin** (Real human reviews from **Yelp**, and synthetic fake reviews from **Gemini** or template **Mocks**). We explore class balances, review lengths, uppercase ratios, and punctuation patterns to identify the unique footprints left by AI generators compared to humans.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 1. Load Data
We load the merged and preprocessed dataset containing the raw text, cleaned text, and extracted structural features.

In [ ]:
processed_data_path = Path("../data/processed/fake_reviews_processed.parquet")

if not processed_data_path.exists():
    # Fallback to local running directory if executed from repo root
    processed_data_path = Path("data/processed/fake_reviews_processed.parquet")

if not processed_data_path.exists():
    raise FileNotFoundError(f"Processed dataset not found at {processed_data_path}. Please run 'uv run python -m src.pipeline' first.")

df = pd.read_parquet(processed_data_path)
print(f"Dataset successfully loaded!")
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

## 2. Review Segmentation by Origin
Let's visualize the proportions of review origins in the dataset: **Yelp** (Genuine Human), **Gemini** (Synthetic API-generated), and **Mock** (Template-based fallback).

In [ ]:
origin_counts = df["origin"].value_counts()
origin_percentages = df["origin"].value_counts(normalize=True) * 100

print("Review Count by Origin:")
for origin, count in origin_counts.items():
    print(f"  {origin}: {count:,} ({origin_percentages[origin]:.2f}%)")

plt.figure(figsize=(14, 5))

# Bar plot of counts
plt.subplot(1, 2, 1)
ax = sns.barplot(x=origin_counts.index, y=origin_counts.values, hue=origin_counts.index, palette="Set2", legend=False)
plt.title("Volume of Reviews by Origin", fontsize=13, fontweight="bold")
plt.ylabel("Count")
plt.xlabel("Origin")
for p in ax.patches:
    ax.annotate(f'{p.get_height():,.0f}', (p.get_x() + p.get_width() / 2., p.get_height() * 0.9),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', color='white', fontweight='bold')

# Pie chart of proportions
plt.subplot(1, 2, 2)
plt.pie(origin_counts.values, labels=origin_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("Set2", len(origin_counts)))
plt.title("Proportion of Review Origins", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

## 3. Structural Feature Analysis by Origin
Let's compare the distributions of review length, capitalization style, and punctuation ratios grouped by review **Origin**. This highlights how human writing styles (Yelp) compare to LLM-generated styles (Gemini) and simple templates (Mock).

In [ ]:
# Compute average values for features grouped by origin
feature_cols = ["char_count", "word_count", "avg_word_length", "cap_ratio", "exclamation_ratio", "question_ratio"]
grouped_stats = df.groupby("origin")[feature_cols].mean().round(4)
print("Average feature values by review origin:")
grouped_stats

### 3.1 Review Length (Word Count) by Origin
How does the length of human-written Yelp reviews compare to Gemini and template-generated reviews?

In [ ]:
plt.figure(figsize=(14, 5))

# Density curve comparison
plt.subplot(1, 2, 1)
limit_words = df["word_count"].quantile(0.95)  # Cap at 95th percentile for clean plot
sns.kdeplot(data=df[df["word_count"] <= limit_words], x="word_count", hue="origin", fill=True, common_norm=False, palette="Set2")
plt.title("Word Count Density by Origin (95th percentile)")
plt.xlabel("Word Count")

# Box plot comparison
plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="origin", y="word_count", palette="Set2", hue="origin", legend=False)
plt.title("Word Count Boxplot by Origin (Log Scale)")
plt.xlabel("Origin")
plt.ylabel("Word Count")
plt.yscale("log")

plt.tight_layout()
plt.show()

### 3.2 Capitalization Style (cap_ratio) by Origin
Capitalization ratio measures the percentage of uppercase letters in the review. It captures stylistic quirks (e.g. dramatic shouting, grammatical correctness, or rigid formatting).

In [ ]:
plt.figure(figsize=(14, 5))

# Density curve comparison
plt.subplot(1, 2, 1)
limit_cap = df["cap_ratio"].quantile(0.99)
sns.kdeplot(data=df[df["cap_ratio"] <= limit_cap], x="cap_ratio", hue="origin", fill=True, common_norm=False, palette="Set2")
plt.title("Uppercase Ratio Density by Origin")
plt.xlabel("Capital Ratio")

# Box plot comparison
plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="origin", y="cap_ratio", palette="Set2", hue="origin", legend=False)
plt.title("Uppercase Ratio Boxplot by Origin")
plt.xlabel("Origin")
plt.ylabel("Capital Ratio")

plt.tight_layout()
plt.show()

### 3.3 Exclamation Mark Ratio by Origin
Exclamation mark intensity is often higher in spam reviews (representing inflated emotional tones). Let's see if this footprint separates Yelp from Gemini and Mock.

In [ ]:
plt.figure(figsize=(14, 5))

# Density curve comparison
plt.subplot(1, 2, 1)
limit_excl = df["exclamation_ratio"].quantile(0.99)
sns.kdeplot(data=df[df["exclamation_ratio"] <= limit_excl], x="exclamation_ratio", hue="origin", fill=True, common_norm=False, palette="Set2")
plt.title("Exclamation Ratio Density by Origin")
plt.xlabel("Exclamation Ratio")

# Box plot comparison
plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="origin", y="exclamation_ratio", palette="Set2", hue="origin", legend=False)
plt.title("Exclamation Ratio Boxplot by Origin")
plt.xlabel("Origin")
plt.ylabel("Exclamation Ratio")

plt.tight_layout()
plt.show()

## 4. Vocabulary & Vocabulary Overlaps by Origin

Let's look at the top 10 most common words used in each segment. This helps us see if the synthetic models use different vocabularies or if they tend to repeat identical terms (overfitting to particular phrases).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_n_words_by_origin(df_data, origin_label, n=10):
    corpus = df_data[df_data["origin"] == origin_label]["clean_text"]
    if len(corpus) == 0:
        return pd.DataFrame(columns=["Word", "Frequency"])
    
    vec = CountVectorizer(stop_words='english').fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)
    return pd.DataFrame(words_freq[:n], columns=['Word', 'Frequency'])

origins_present = df["origin"].unique()
plt.figure(figsize=(18, 5))

colors = {"Yelp": "skyblue", "Gemini": "salmon", "Mock": "lightgreen"}

for i, origin in enumerate(origins_present):
    plt.subplot(1, len(origins_present), i+1)
    top_words = get_top_n_words_by_origin(df, origin, n=10)
    if len(top_words) > 0:
        sns.barplot(data=top_words, x='Frequency', y='Word', color=colors.get(origin, "gray"))
        plt.title(f'Top Words in {origin}')
        plt.xlabel('Frequency')
        
plt.tight_layout()
plt.show()

## 5. Key Findings for the Restaurant Model

1. **Origin Proportions**: Real human Yelp reviews make up the majority of our genuine base, while Gemini and/or template Mocks form the fake class. 
2. **Stylistic Footprints**:
   * **Word Count**: Gemini generated reviews might concentrate strongly in a narrow length distribution (e.g. 20-40 words) due to our prompt instructions, whereas human Yelp reviews show a highly spread, long-tailed distribution.
   * **Capitalization**: Check if Gemini's capital ratios are highly consistent (due to correct punctuation and grammar), whereas human reviews exhibit higher variance (some all lowercase, some all caps).
   * **Exclamation Marks**: Synthetic templates (Mock) and Gemini might show higher concentrations of exclamation marks due to simulated glowing/sabotage sentiment templates.
3. **Vocabulary Patterns**: Examine if the top words in Gemini/Mock are focused heavily on the prompt inputs (like the restaurant names or dishes we generated), while Yelp is much broader.